In [5]:
import pandas as pd
import numpy as np
import os

def convert_to_svsig(inpath, filename, cohort_name, outpath):

    def chr_to_num(x):
        if x == "X":
            return 23
        if x == "Y":
            return 24
        return int(float(x))

    def is_mito(x):
        return x=="M"

    orig_SV_list_df = pd.read_table(inpath+filename)
    
    mask_mito = orig_SV_list_df['chr1'].apply(is_mito) | orig_SV_list_df['chr2'].apply(is_mito)
    orig_SV_list_df = orig_SV_list_df.loc[~mask_mito].reset_index(drop=True)

    seqnames_list = list(orig_SV_list_df['chr1'].map(chr_to_num))
    start_list = list(orig_SV_list_df['pos1'])
    strand_list = list(map(lambda x: '+' if x == 0 else '-', list(orig_SV_list_df['str1'])))
    altchr_list = list(orig_SV_list_df['chr2'].map(chr_to_num))
    altpos_list = list(orig_SV_list_df['pos2'])
    altstrand_list = list(map(lambda x: '+' if x == 0 else '-', list(orig_SV_list_df['str2'])))
    dcc_project_code_list = list(map(lambda x: cohort_name, range(orig_SV_list_df.shape[0])))
    sv_id_list = list(np.arange(orig_SV_list_df.shape[0]))
    sid_list = list(orig_SV_list_df['individual'])
    donor_unique_id_list = list(orig_SV_list_df['individual'])
    weights_list = list(np.ones(orig_SV_list_df.shape[0]))
    topo_list = list(np.zeros(orig_SV_list_df.shape[0]))
    topo_n_list = list(np.zeros(orig_SV_list_df.shape[0]))
    mech_list = list(np.zeros(orig_SV_list_df.shape[0]))
    homseq_list = list(np.zeros(orig_SV_list_df.shape[0]))

    d = {'seqnames': seqnames_list,
		'start': start_list,
		'strand': strand_list,
		'altchr': altchr_list,
		'altpos': altpos_list,
		'altstrand': altstrand_list,
		'dcc_project_code': dcc_project_code_list,
		'sv_id': sv_id_list,
		'sid': sid_list,
		'donor_unique_id': donor_unique_id_list,
		'weights': weights_list,
		'topo': topo_list,
		'topo_n': topo_n_list,
		'mech': mech_list,
		'homseq': homseq_list,
		}

    SV_list_for_SVsig_df = pd.DataFrame(data=d)
    SV_list_for_SVsig_df.to_csv(outpath + filename[:-4] + "_svsig.csv", sep = ",", index = False)

In [ ]:
import os

inpath = '~/Documents/Getz Lab/1. TCGA-WGS/Data/NEW_filtered/'
suffix = 'PRAD.new.merged.filtered.SV.tsv'
outpath = '~/SVsig-antoniakowalewski/data/input_data/'

for filename in os.listdir(inpath):
    if filename.endswith(suffix):
        cohort_name = filename.split('.')[0]
        convert_to_svsig(inpath, filename, cohort_name, outpath)